# 实验二：房源采集与 MySQL 存储

- 班级：数据1241
- 姓名：杨素素
- 学号：112404260105

## 运行顺序
1. 安装依赖并执行“公共设置”。
2. 请求第一页，分析 `div.property` 和 8 个字段。
3. 运行分页采集，生成 `data/houseinfo.csv` 和 `data/page_logs.json`。
4. 回读 CSV，检查 498 行、8 列和三条样本。
5. 运行 MySQL 入库单元，随后再次运行，确认新增条数为 0。
6. 运行三条样本核对单元，再到 MySQL 终端截图。


## 0. 安装依赖（只需运行一次）

In [24]:
%pip install requests beautifulsoup4 pymysql

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. 公共设置

下面的路径以 Notebook 所在目录 `lab02` 为基准。运行后应能看到 `base_dir` 指向 `.../Lab02`。

In [28]:
import csv
import json
import socket
import time
from pathlib import Path
from urllib.parse import urljoin, urlparse

import pymysql
import requests
from bs4 import BeautifulSoup


BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "data"
CSV_PATH = DATA_DIR / "houseinfo.csv"
LOG_PATH = DATA_DIR / "page_logs.json"
DATA_DIR.mkdir(exist_ok=True)

FIELDS = ["title", "house", "address", "struct", "area", "floor", "maketime", "price"]

print("base_dir =", BASE_DIR)
print("csv_path =", CSV_PATH)
print("log_path =", LOG_PATH)

base_dir = d:\大数据采集与处理-学生实验环境\大数据采集与处理-学生实验环境\workspace\Lab02
csv_path = d:\大数据采集与处理-学生实验环境\大数据采集与处理-学生实验环境\workspace\Lab02\data\houseinfo.csv
log_path = d:\大数据采集与处理-学生实验环境\大数据采集与处理-学生实验环境\workspace\Lab02\data\page_logs.json


## 2. 请求第一页并检查页面

如果默认域名返回“访问验证”，程序会通过 IPv4 地址访问同一教学站，并保留 `Host: www.bspider.top`。

> **运行后先检查，不作为报告必交截图**：输出应包含 `status=200`、正常的房源页面标题和 `property_count=60`。如果 `property_count=0`，说明拿到的是访问验证页或页面结构已变化，先不要继续运行分页采集。

In [29]:
START_URL = "http://www.bspider.top/anjuke/"
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/142.0.0.0 Safari/537.36"
    )
}

# 一般留空即可；若域名解析异常，可临时填入教师确认的 IPv4 地址。
IPV4_OVERRIDE = ""


def request_page(url):
    parsed = urlparse(url)
    path = parsed.path + (f"?{parsed.query}" if parsed.query else "")
    ipv4 = IPV4_OVERRIDE.strip() or socket.gethostbyname("www.bspider.top")
    request_url = f"http://{ipv4}{path}"
    response = requests.get(
        request_url,
        headers={**HEADERS, "Host": parsed.netloc, "Connection": "close"},
        timeout=(10, 30),
    )
    response.raise_for_status()
    response.encoding = "utf-8"
    print(
        f"request_url={request_url} status={response.status_code} "
        f"bytes={len(response.content)}"
    )
    return response.text


html = request_page(START_URL)
soup = BeautifulSoup(html, "html.parser")
property_count = len(soup.select("div.property"))

print("page_title =", soup.title.get_text(strip=True) if soup.title else "")
print("property_count =", property_count)

if property_count == 0:
    raise RuntimeError("没有解析到房源，请检查是否返回了访问验证页")

request_url=http://172.67.199.181/anjuke/ status=200 bytes=291926
page_title = 重庆二手房房产网，重庆二手房出售，重庆买房购房交易信息 - 58安居客
property_count = 60


## 3. 查看房源父容器和字段节点

`rows = soup.select("div.property")` 取得整个页面的房源列表；`row.select_one(...)` 只在当前一条房源中查找字段。

In [31]:
rows = soup.select("div.property")
row = rows[0]

title_node = row.select_one("h3.property-content-title-name")
house_node = row.select_one("p.property-content-info-comm-name")
address_node = row.select_one("p.property-content-info-comm-address")
struct_node = row.select_one("p.property-content-info-attribute")
price_node = row.select_one("p.property-price-average")

detail_nodes = row.select(
    "div.property-content-info:not(.property-content-info-comm) "
    "> p.property-content-info-text:not(.property-content-info-attribute)"
)

print("rows =", len(rows))
print("row_class =", row.get("class"))
print("title =", title_node.get_text(" ", strip=True))
print("house =", house_node.get_text(" ", strip=True))
print("address =", address_node.get_text("", strip=True))
print("struct =", struct_node.get_text("", strip=True))
print("price =", price_node.get_text(" ", strip=True))

print("\n属性列表：")
for index, node in enumerate(detail_nodes, start=1):
    print(index, node.get_text(" ", strip=True))

rows = 60
row_class = ['property']
title = 万科物业自营 南坪六公里金域翰林 三号线精装正三房带家具电器
house = 万科金域学府翰林菁华里
address = 南岸学府大道二塘路59号
struct = 3室2厅1卫
price = 16465元/㎡

属性列表：
1 89.9㎡
2 南北
3 低层(共27层)
4 2019年建造


## 4. 编写 8 字段解析函数

面积按 `㎡` 后缀识别，楼层按“层”识别，建造年代按“年建造”识别，因此不依赖固定的列表下标。

In [35]:
def parse_page(html):
    soup = BeautifulSoup(html, "html.parser")
    rows = soup.select("div.property")
    records = []

    for row in rows:
        title_node = row.select_one("h3.property-content-title-name")
        house_node = row.select_one("p.property-content-info-comm-name")
        address_node = row.select_one("p.property-content-info-comm-address")
        struct_node = row.select_one("p.property-content-info-attribute")
        price_node = row.select_one("p.property-price-average")
        detail_nodes = row.select(
            "div.property-content-info:not(.property-content-info-comm) "
            "> p.property-content-info-text:not(.property-content-info-attribute)"
        )
        details = [node.get_text(" ", strip=True) for node in detail_nodes]

        area = next((value for value in details if value.endswith("㎡")), "")
        floor = next((value for value in details if "层" in value), "")
        maketime = next((value for value in details if value.endswith("年建造")), "")
        if not area:
            raise ValueError(f"未找到面积字段: {details!r}")

        records.append(
            [
                title_node.get_text(" ", strip=True) if title_node else "",
                house_node.get_text(" ", strip=True) if house_node else "",
                address_node.get_text("", strip=True) if address_node else "",
                struct_node.get_text("", strip=True) if struct_node else "",
                area,
                floor,
                maketime,
                price_node.get_text(" ", strip=True) if price_node else "",
            ]
        )

    pagination = soup.select_one("div.pagination")
    next_active = soup.select_one("a.next.next-active")
    disabled_next = (
        pagination.select_one("a.next:not(.next-active)") if pagination else None
    )
    page_title = soup.title.get_text(strip=True) if soup.title else ""
    next_href = next_active.get("href") if next_active else None
    return records, next_href, page_title, pagination is not None, disabled_next

## 5. 验证第一条房源

运行后应在输出中看到 8 个字段，顺序为标题、楼盘、地址、户型、面积、楼层、建造年代、单价。

In [36]:
records, next_href, page_title, pagination_exists, disabled_next = parse_page(html)
first_record = records[0]

print("page_title =", page_title)
print("page_records =", len(records))
print("next_href =", next_href)
print("pagination_exists =", pagination_exists)

print("\n第一条房源：")
for field, value in zip(FIELDS, first_record, strict=True):
    print(f"{field:8} = {value}")

page_title = 重庆二手房房产网，重庆二手房出售，重庆买房购房交易信息 - 58安居客
page_records = 60
next_href = ?page=2
pagination_exists = True

第一条房源：
title    = 万科物业自营 南坪六公里金域翰林 三号线精装正三房带家具电器
house    = 万科金域学府翰林菁华里
address  = 南岸学府大道二塘路59号
struct   = 3室2厅1卫
area     = 89.9㎡
floor    = 低层(共27层)
maketime = 2019年建造
price    = 16465元/㎡


## 6. 编写分页采集函数

分页函数会保存 `page_logs.json`，按 8 个字段完全去重，并用 `utf-8-sig` 写入 CSV。

末页判断不是“找不到按钮就结束”，而是分三步检查：

1. `div.pagination` 必须存在。
2. 下一页节点必须存在，但不能再带 `next-active`。
3. 禁用节点的 `href` 必须为空或 `#`。


In [41]:
def crawl(max_pages=20):
    current_url = START_URL
    visited = set()
    all_records = []
    page_logs = []

    for page_no in range(1, max_pages + 1):
        if current_url in visited:
            raise RuntimeError(f"检测到重复页面地址: {current_url}")
        visited.add(current_url)

        html = request_page(current_url)
        records, next_href, page_title, pagination_exists, disabled_next = parse_page(html)
        if not records:
            raise RuntimeError(f"第 {page_no} 页没有解析到房源，页面标题: {page_title}")

        disabled_href = disabled_next.get("href") if disabled_next else None
        all_records.extend(records)
        page_logs.append(
            {
                "page": page_no,
                "url": current_url,
                "count": len(records),
                "total": len(all_records),
                "title": page_title,
                "next_href": next_href,
                "pagination_exists": pagination_exists,
                "disabled_next_href": disabled_href,
            }
        )
        print(
            f"page={page_no} url={current_url} "
            f"count={len(records)} total={len(all_records)}"
        )

        if not next_href:
            if not pagination_exists:
                raise RuntimeError(f"第 {page_no} 页分页区域消失，结构异常")
            if disabled_next is None:
                raise RuntimeError(f"第 {page_no} 页找不到下一页按钮，结构异常")
            if disabled_href not in (None, "", "#"):
                raise RuntimeError(
                    f"第 {page_no} 页下一页按钮状态异常: {disabled_href!r}"
                )
            print(
                "stop=分页区域存在，下一页按钮已禁用，"
                f"disabled_href={disabled_href!r}"
            )
            break

        next_url = urljoin(current_url, next_href)
        if urlparse(next_url).netloc != urlparse(START_URL).netloc:
            raise RuntimeError(f"下一页越出教学站: {next_url}")
        if next_url in visited:
            raise RuntimeError(f"下一页地址已访问过: {next_url}")

        current_url = next_url
        time.sleep(1)
    else:
        raise RuntimeError("达到安全页数上限，采集中止")

    deduped = []
    seen = set()
    for record in all_records:
        key = tuple(record)
        if key not in seen:
            seen.add(key)
            deduped.append(record)

    with CSV_PATH.open("w", encoding="utf-8-sig", newline="") as file:
        writer = csv.writer(file)
        writer.writerow(FIELDS)
        writer.writerows(deduped)

    with LOG_PATH.open("w", encoding="utf-8") as file:
        json.dump(page_logs, file, ensure_ascii=False, indent=2)

    print(
        f"collected={len(all_records)} deduped={len(deduped)} "
        f"removed={len(all_records) - len(deduped)}"
    )
    return all_records, deduped, page_logs

## 7. 执行分页采集

运行本单元会实际访问 10 页，大约需要十几秒。完成后查看 `data/houseinfo.csv` 和 `data/page_logs.json`。

In [42]:
all_records, deduped, page_logs = crawl()

print("\n最后三页日志：")
for item in page_logs[-3:]:
    print(item)

request_url=http://172.67.199.181/anjuke/ status=200 bytes=291926
page=1 url=http://www.bspider.top/anjuke/ count=60 total=60
request_url=http://172.67.199.181/anjuke/?page=2 status=200 bytes=296488
page=2 url=http://www.bspider.top/anjuke/?page=2 count=60 total=120
request_url=http://172.67.199.181/anjuke/?page=3 status=200 bytes=297907
page=3 url=http://www.bspider.top/anjuke/?page=3 count=60 total=180
request_url=http://172.67.199.181/anjuke/?page=4 status=200 bytes=298113
page=4 url=http://www.bspider.top/anjuke/?page=4 count=60 total=240
request_url=http://172.67.199.181/anjuke/?page=5 status=200 bytes=298268
page=5 url=http://www.bspider.top/anjuke/?page=5 count=60 total=300
request_url=http://172.67.199.181/anjuke/?page=6 status=200 bytes=301690
page=6 url=http://www.bspider.top/anjuke/?page=6 count=60 total=360
request_url=http://172.67.199.181/anjuke/?page=7 status=200 bytes=300630
page=7 url=http://www.bspider.top/anjuke/?page=7 count=60 total=420
request_url=http://172.67.19

## 8. 回读 CSV 并核对

In [43]:
def read_csv():
    with CSV_PATH.open(encoding="utf-8-sig", newline="") as file:
        reader = csv.DictReader(file)
        if reader.fieldnames != FIELDS:
            raise ValueError(f"CSV 表头不符合要求: {reader.fieldnames!r}")

        rows = []
        for line_no, row in enumerate(reader, start=2):
            values = [row[field] for field in FIELDS]
            if len(values) != 8:
                raise ValueError(f"CSV 第 {line_no} 行不是 8 个字段")
            rows.append(values)
    return rows


csv_rows = read_csv()
unique_rows = {tuple(row) for row in csv_rows}
csv_samples = [csv_rows[0], csv_rows[60], csv_rows[-1]]

print("headers =", FIELDS)
print("csv_rows =", len(csv_rows))
print("columns =", len(FIELDS))
print("unique_rows =", len(unique_rows))
print("duplicates_within_csv =", len(csv_rows) - len(unique_rows))

for position, sample in zip([1, 61, len(csv_rows)], csv_samples, strict=True):
    print(f"\nsample={position}")
    print(" | ".join(sample))

headers = ['title', 'house', 'address', 'struct', 'area', 'floor', 'maketime', 'price']
csv_rows = 498
columns = 8
unique_rows = 498
duplicates_within_csv = 0

sample=1
万科物业自营 南坪六公里金域翰林 三号线精装正三房带家具电器 | 万科金域学府翰林菁华里 | 南岸学府大道二塘路59号 | 3室2厅1卫 | 89.9㎡ | 低层(共27层) | 2019年建造 | 16465元/㎡

sample=61
万科物业自营 朝中庭 业主诚心出售 | 万科金域学府翰林菁华里 | 南岸学府大道二塘路59号 | 3室2厅1卫 | 85.7㎡ | 低层(共27层) | 2018年建造 | 16914元/㎡

sample=498
恒大照母山 精装3房 内圈位置 户型方正 拎包即住 诚意出售 | 恒大照母山山水城 | 渝北照母山恒康路9号 | 3室2厅1卫 | 90.2㎡ | 低层(共32层) | 2015年建造 | 15530元/㎡


## 9. 编写 MySQL 入库函数

本单元只读取第二阶段的 CSV，不会重新采集网页。8 个 `%s` 依次对应 8 个房源字段。

In [45]:
DB_CONFIG = {
    "host": "mysql",
    "port": 3306,
    "database": "smartpark_lab",
    "user": "student",
    "password": "123456",
    "charset": "utf8mb4",
    "autocommit": False,
}


def load_to_mysql(rows):
    insert_sql = """
        INSERT INTO houseinfo
            (title, house, address, struct, area, floor, maketime, price)
        VALUES
            (%s, %s, %s, %s, %s, %s, %s, %s)
    """
    conn = None
    cursor = None

    try:
        conn = pymysql.connect(**DB_CONFIG)
        cursor = conn.cursor()
        cursor.execute("SELECT COUNT(*) FROM houseinfo")
        before_count = cursor.fetchone()[0]

        cursor.execute(
            "SELECT title, house, address, struct, area, floor, maketime, price "
            "FROM houseinfo"
        )
        existing = {tuple(row) for row in cursor.fetchall()}
        missing = [row for row in rows if tuple(row) not in existing]

        cursor.executemany(insert_sql, missing)
        conn.commit()

        cursor.execute("SELECT COUNT(*) FROM houseinfo")
        after_count = cursor.fetchone()[0]
        print(
            f"before_count={before_count} csv_rows={len(rows)} "
            f"inserted={len(missing)} after_count={after_count}"
        )
        return before_count, len(missing), after_count
    except Exception:
        if conn is not None:
            conn.rollback()
        raise
    finally:
        if cursor is not None:
            cursor.close()
        if conn is not None:
            conn.close()

## 10. 首次入库

In [46]:
load_to_mysql(csv_rows)

OperationalError: (2003, "Can't connect to MySQL server on 'mysql' ([Errno 11001] getaddrinfo failed)")

## 11. 在 Notebook 中核对三条样本


In [15]:
def query_samples(rows):
    sample_indexes = [0, 60, len(rows) - 1]
    conn = pymysql.connect(**DB_CONFIG)
    cursor = conn.cursor()

    try:
        for index in sample_indexes:
            expected = tuple(rows[index])
            cursor.execute(
                "SELECT title, house, address, struct, area, floor, maketime, price "
                "FROM houseinfo WHERE title=%s AND house=%s AND address=%s "
                "AND struct=%s AND area=%s AND floor=%s AND maketime=%s AND price=%s",
                expected,
            )
            found = cursor.fetchone()
            print(
                f"sample={index + 1} matched={found is not None} "
                f"area={expected[4]} price={expected[7]} maketime={expected[6]}"
            )
            print(" | ".join(expected))
    finally:
        cursor.close()
        conn.close()


query_samples(csv_rows)

sample=1 matched=True area=89.9㎡ price=16465元/㎡ maketime=2019年建造
万科物业自营 南坪六公里金域翰林 三号线精装正三房带家具电器 | 万科金域学府翰林菁华里 | 南岸学府大道二塘路59号 | 3室2厅1卫 | 89.9㎡ | 低层(共27层) | 2019年建造 | 16465元/㎡
sample=61 matched=True area=85.7㎡ price=16914元/㎡ maketime=2018年建造
万科物业自营 朝中庭 业主诚心出售 | 万科金域学府翰林菁华里 | 南岸学府大道二塘路59号 | 3室2厅1卫 | 85.7㎡ | 低层(共27层) | 2018年建造 | 16914元/㎡
sample=498 matched=True area=90.2㎡ price=15530元/㎡ maketime=2015年建造
恒大照母山 精装3房 内圈位置 户型方正 拎包即住 诚意出售 | 恒大照母山山水城 | 渝北照母山恒康路9号 | 3室2厅1卫 | 90.2㎡ | 低层(共32层) | 2015年建造 | 15530元/㎡


## 12. MySQL 终端截图清单

回到 `mysql-console.bat`，依次执行并截图：

```sql
USE smartpark_lab;
SHOW TABLES;
DESCRIBE houseinfo;
SELECT COUNT(*) AS before_count FROM houseinfo;
```

首次入库后执行：

```sql
SELECT COUNT(*) AS after_count FROM houseinfo;
SELECT title, house, address, struct, area, floor, maketime, price
FROM houseinfo ORDER BY id LIMIT 5;
```

重跑后再次执行：

```sql
SELECT COUNT(*) AS after_rerun_count FROM houseinfo;
```